# CREAM Noisy-Graph Analysis
This notebook compares **CREAM with ground-truth DAG** vs **CREAM with noisy DAGs** for:
- `CelebA`
- `Complete_Concept_FMNIST`

Workflow:
1. Locate experiment results CSVs
2. Visualize ground-truth vs noisy DAGs
3. Compare metrics (`test_task_accuracy`, `test_concept_accuracy`, `CCI`, `concept_leakage`)
4. Summarize GT-vs-noisy deltas per dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import display
import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", font_scale=1.0)
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 110

# Try server path first, then local workspace path
EXPERIMENTS_ROOT_CANDIDATES = [
    Path('/home/dani00003/mCREAM/experiments'),
    Path(r'c:/Daten/D_Part/Personal/mCREAM/experiments'),
]
EXPERIMENTS_ROOT = next((p for p in EXPERIMENTS_ROOT_CANDIDATES if p.exists()), EXPERIMENTS_ROOT_CANDIDATES[-1])

PROJECT_ROOT = EXPERIMENTS_ROOT.parent if EXPERIMENTS_ROOT.name == 'experiments' else Path(r'c:/Daten/D_Part/Personal/mCREAM')

DATASET_SPECS = {
    'CelebA': {
        'model_folder': 'Standard_CelebA',
        'ground_truth_dag': PROJECT_ROOT / 'data/CelebA/final_DAG_unfair.csv',
        'noisy_dir': PROJECT_ROOT / 'data/CelebA/noisy_dags',
    },
    'Complete_Concept_FMNIST': {
        'model_folder': 'Standard_FashionMNIST',
        'ground_truth_dag': PROJECT_ROOT / 'data/FashionMNIST/Complete_Concept_FMNIST_DAG.csv',
        'noisy_dir': PROJECT_ROOT / 'data/FashionMNIST/noisy_dags',
    },
}

NOISY_LEVELS = ['low', 'medium', 'high', 'structured_bias']
METRICS = [
    'test_task_accuracy',
    'test_concept_accuracy',
    'CCI',
    'PFI_concept_importance',
    'PFI_side_importance',
    'concept_leakage',
    'c2y_baseline_accuracy',
]

print(f'Using EXPERIMENTS_ROOT: {EXPERIMENTS_ROOT}')
print(f'Exists: {EXPERIMENTS_ROOT.exists()}')
print(f'Using PROJECT_ROOT: {PROJECT_ROOT}')

## 1) Ground-Truth vs Noisy Graphs
This section shows the ground-truth DAG first, then each noisy DAG level used in CREAM-noisy experiments.

In [ ]:
def read_dag_csv(path: Path):
    if not path.exists():
        return None
    return pd.read_csv(path, index_col=0)

def plot_gt_vs_noisy_for_dataset(dataset_name: str):
    spec = DATASET_SPECS[dataset_name]
    gt_path = spec['ground_truth_dag']
    noisy_dir = spec['noisy_dir']

    gt_df = read_dag_csv(gt_path)
    if gt_df is None:
        print(f'[MISS] Ground truth DAG not found: {gt_path}')
        return

    noisy_paths = {
        lvl: noisy_dir / f'noisy_dag_{lvl}.csv' for lvl in NOISY_LEVELS
    }

    cols = 1 + len(NOISY_LEVELS)
    fig, axes = plt.subplots(1, cols, figsize=(4*cols, 4), squeeze=False)
    axes = axes[0]

    sns.heatmap(gt_df.values.astype(float), ax=axes[0], vmin=0, vmax=1, cbar=False, cmap='RdYlGn')
    axes[0].set_title(f'{dataset_name}\nGround Truth')
    axes[0].set_xlabel('')
    axes[0].set_ylabel('')

    for i, lvl in enumerate(NOISY_LEVELS, start=1):
        dag_df = read_dag_csv(noisy_paths[lvl])
        if dag_df is None:
            axes[i].axis('off')
            axes[i].set_title(f'{lvl}\nmissing')
            continue

        sns.heatmap(dag_df.values.astype(float), ax=axes[i], vmin=0, vmax=1, cbar=False, cmap='RdYlGn')
        diff_pct = (dag_df.values.astype(bool) != gt_df.values.astype(bool)).mean() * 100
        axes[i].set_title(f'Noisy: {lvl}\nΔ edges: {diff_pct:.1f}%')
        axes[i].set_xlabel('')
        axes[i].set_ylabel('')

    plt.tight_layout()
    plt.show()

for ds in DATASET_SPECS.keys():
    plot_gt_vs_noisy_for_dataset(ds)

## 2) Load CREAM Ground-Truth and CREAM-Noisy Results
This section scans `experiments/<dataset>/train_cbm/<model>/.../last_metrics/*.csv` and labels each run as:
- `ground_truth` (CREAM baseline configs)
- `noisy` (CREAM noisy configs)

In [ ]:
def infer_condition_from_filename(stem: str):
    s = stem.lower()
    if 'cream_noisy' in s:
        level = next((lvl for lvl in NOISY_LEVELS if lvl in s), None)
        return 'noisy', level
    if 'cream_best' in s or 'cream_cgm_pdag' in s:
        return 'ground_truth', 'ground_truth'
    return None, None

def load_cream_gt_noisy_results(experiments_root: Path):
    rows = []

    for dataset_name, spec in DATASET_SPECS.items():
        model_folder = spec['model_folder']
        base = experiments_root / dataset_name / 'train_cbm' / model_folder
        if not base.exists():
            print(f'[SKIP] Missing folder: {base}')
            continue

        for csv_f in base.rglob('last_metrics/*.csv'):
            stem = csv_f.stem
            condition, noisy_level = infer_condition_from_filename(stem)
            if condition is None:
                continue

            try:
                df = pd.read_csv(csv_f)
                if len(df) == 0:
                    continue
                row = df.iloc[0].to_dict()
                row['dataset'] = dataset_name
                row['condition'] = condition
                row['noisy_level'] = noisy_level
                row['config_name'] = stem
                row['source_file'] = str(csv_f)
                rows.append(row)
            except Exception as exc:
                print(f'[ERR] {csv_f}: {exc}')

    if not rows:
        return pd.DataFrame()

    out = pd.DataFrame(rows)
    return out

def make_summary_table(df, metric_cols, group_cols):
    available = [c for c in metric_cols if c in df.columns]
    if not available:
        return pd.DataFrame()

    grouped = df.groupby(group_cols)[available]
    means = grouped.mean(numeric_only=True)
    stds = grouped.std(numeric_only=True).fillna(0.0)
    counts = grouped.size().rename('n_runs')

    summary = pd.DataFrame(index=means.index)
    summary['n_runs'] = counts
    for col in available:
        summary[col] = means[col].map(lambda x: f'{x:.4f}') + ' ± ' + stds[col].map(lambda x: f'{x:.4f}')
    return summary

results_df = load_cream_gt_noisy_results(EXPERIMENTS_ROOT)
print(f'Loaded rows: {len(results_df)}')
if len(results_df) > 0:
    print('Datasets:', sorted(results_df['dataset'].unique()))
    print('Conditions:', sorted(results_df['condition'].unique()))
    print('Noisy levels:', sorted(results_df['noisy_level'].dropna().unique()))
    display(results_df[['dataset','condition','noisy_level','config_name','source_file']].head(20))

## 3) Metric Tables: Ground-Truth vs Noisy (Both Datasets)

In [ ]:
if len(results_df) == 0:
    print('No CREAM GT/Noisy rows found. Check EXPERIMENTS_ROOT and finished runs.')
else:
    summary = make_summary_table(
        results_df,
        metric_cols=METRICS,
        group_cols=['dataset', 'condition', 'noisy_level']
    )
    print('=' * 90)
    print('CREAM: GROUND-TRUTH vs NOISY SUMMARY')
    print('=' * 90)
    display(summary)

    # Also show raw means for easier delta calculations
    raw_means = (
        results_df.groupby(['dataset','condition','noisy_level'])[METRICS]
        .mean(numeric_only=True)
        .reset_index()
    )
    display(raw_means)

## 4) Plots: Accuracy and Reliance Deltas (GT baseline)

In [ ]:
if len(results_df) == 0:
    print('No data to plot.')
else:
    metrics_to_plot = ['test_task_accuracy', 'test_concept_accuracy', 'CCI', 'concept_leakage']
    available = [m for m in metrics_to_plot if m in results_df.columns]

    # Boxplots by condition/noisy level
    for ds in sorted(results_df['dataset'].unique()):
        subset = results_df[results_df['dataset'] == ds].copy()
        if subset.empty:
            continue

        subset['group'] = np.where(
            subset['condition'] == 'ground_truth',
            'ground_truth',
            subset['noisy_level']
        )

        fig, axes = plt.subplots(1, len(available), figsize=(5 * len(available), 4), squeeze=False)
        axes = axes[0]
        for ax, metric in zip(axes, available):
            plot_df = subset.dropna(subset=[metric])
            if plot_df.empty:
                ax.set_title(f'{metric}\n(no data)')
                continue
            order = ['ground_truth'] + [lvl for lvl in NOISY_LEVELS if lvl in plot_df['group'].values]
            sns.boxplot(data=plot_df, x='group', y=metric, ax=ax, order=order)
            ax.set_title(f'{ds}\n{metric}')
            ax.tick_params(axis='x', rotation=20)
            if metric == 'concept_leakage':
                ax.axhline(0, color='red', linestyle='--', alpha=0.5)
        plt.tight_layout()
        plt.show()

    # Delta table: noisy_mean - ground_truth_mean per dataset
    mean_df = results_df.groupby(['dataset', 'condition', 'noisy_level'])[available].mean(numeric_only=True).reset_index()
    delta_rows = []
    for ds in sorted(mean_df['dataset'].unique()):
        gt = mean_df[(mean_df['dataset'] == ds) & (mean_df['condition'] == 'ground_truth')]
        if gt.empty:
            continue
        gt_vals = gt.iloc[0]
        noisy_df = mean_df[(mean_df['dataset'] == ds) & (mean_df['condition'] == 'noisy')]
        for _, r in noisy_df.iterrows():
            row = {'dataset': ds, 'noisy_level': r['noisy_level']}
            for m in available:
                row[f'delta_{m}'] = r[m] - gt_vals[m]
            delta_rows.append(row)

    delta_df = pd.DataFrame(delta_rows)
    if len(delta_df) > 0:
        print('=' * 90)
        print('DELTA TABLE (noisy - ground_truth)')
        print('=' * 90)
        display(delta_df.sort_values(['dataset', 'noisy_level']))
    else:
        print('No GT baseline rows found for delta computation.')